> **역할: [발표·초기(최종 아님)]**  (전체 순서·최종은 `NOTEBOOK_INDEX.md` / 최종 모델 nb24)

> ⚠️ **이 발표용 노트북은 초기 버전입니다. 최종 모델은 nb24** 입니다(누설 제거 정직본 기준).

# 🎯 창업 입지 평가 프로젝트 — 후배(1·2학년)를 위한 상세 설명

> 데이터마이닝 수업에서 만든 프로젝트입니다.
> **"가게를 차리고 싶은데 어디가 좋을까?" 를 데이터로 답하는 모델** 을 만들었어요.

---

## 📌 이 발표에서 알게 되는 것

1. **데이터마이닝**이 무엇이고 왜 하는지
2. 우리가 푼 문제는 정확히 무엇인지
3. 데이터를 어디서, 어떻게 가져왔는지
4. 모델은 어떻게 학습하고 무엇을 예측하는지
5. 결과를 어떻게 해석하고, 한계는 무엇인지

---

## 1️⃣ 데이터마이닝이 뭐예요?

**데이터마이닝 = 데이터에서 '쓸 만한 패턴'을 찾아내는 일**

예시:
- 넷플릭스가 "당신이 좋아할 영화"를 추천하는 것
- 유튜브가 다음 영상을 자동 재생하는 것
- 카드사가 "이상한 결제"를 감지하는 것

→ 데이터를 모아 → 가공해서 → 모델에 학습시키면 → 새 입력에 답을 줌

### 우리 프로젝트의 데이터마이닝

```
입력: 자치구 + 업종 + 층수 + 투자금
   ↓ (모델)
출력: 성공 확률 % (0 ~ 100)
```

예: "강남구에서 한식음식점을 1층에 1억원 들여 차리면? → 성공 확률 54%"



## 2️⃣ 왜 이 문제를 풀었나? — 동기

### 자영업의 현실
- 매년 새 가게가 **수십만 개** 열리고, 비슷한 수만큼 닫음
- 통계청 KOSIS: **5년 후 생존율 30% (음식점은 22%)** — 10개 중 7개가 5년 안에 망함 *(출처 미확인 — KOSIS 표 ID/URL 없이 인용, 검증 불가)*
- "어디서, 어떤 가게를 차려야 하나?" 는 인생을 좌우하는 결정

### 시장에 있는 솔루션
| 서비스 | 가격 | 접근성 |
|---|---|---|
| 나이스비즈맵 | 💰 유료 | 일반인 X |
| 소상공인진흥공단 | 무료 | API 키 별도 신청 |
| 서울시 우리마을가게 (golmok.seoul.go.kr) | 무료 | 점수 형태 X — 데이터만 |

→ **공개 데이터만으로 일반인이 쓸 수 있는 입지 평가 모델은 부재**.
→ 우리가 만들기로!

---

## 3️⃣ 어떻게 풀었나? — 전체 흐름

```
       【 원본 데이터 】
          ↓
   ┌──────────────────────┐
   │ 1. 데이터 수집·정리     │  서울시 + 팀원 4명 자료
   │ 2. EDA (탐색)          │  먼저 데이터 모양 보기
   │ 3. 변수 만들기          │  핵심! 모델에 줄 신호 설계
   └──────────────────────┘
          ↓
   ┌──────────────────────┐
   │ 4. 여러 모델 학습       │  강의 범위 모든 알고리즘
   │ 5. 약점 보완            │  Ablation, 시간 분할 검증
   │ 6. 모델 해석            │  무엇이 중요한가
   └──────────────────────┘
          ↓
   ┌──────────────────────┐
   │ 7. 변수 중요도           │  외부 변수 vs 카테고리
   │ 8. 비지도 학습           │  KMeans + PCA
   │ 9. 지도 시각화           │  자치구별 지도
   └──────────────────────┘
          ↓
   ┌──────────────────────┐
   │ 10. 평가기 (자치구)      │  evaluate_location()
   │ 11. 평가기 (좌표·층수)    │  evaluate_startup_point()
   │ 12. 검증·시행착오 기록   │  v1 → v6
   └──────────────────────┘
          ↓
   ┌──────────────────────┐
   │ 13. 최종 결론·시연       │  성공 확률 등급 (A/B/C/D/F)
   │ 14. 심층 검증            │  회귀 R², BEP 민감도
   │ 15. 약점 보완            │  폐업 페널티, BEP 신뢰구간
   │ 16. 서울시 API 비교       │  도메인 검증
   │ 17. 외부 검증            │  폐업·KOSIS·도메인
   └──────────────────────┘
```

---

## 4️⃣ 데이터 — 어디서 가져왔어요?

### 📊 데이터 출처 4가지

| 출처 | 데이터 | 단위 |
|---|---|---|
| **서울 열린데이터광장** | 매출·점포·유동인구·집객시설·변화지표 (9개 API) | 상권/자치구/행정동 × 25분기 (2020Q1~2026Q1) |
| **팀1 (이승직)** | 부동산·임대료·공실률·마진 proxy | 자치구·분기 |
| **팀2 (한승현)** | 가맹점 브랜드별 매출·창업비 | 업종 |
| **팀3** | 행정동 단위 통합 매출 | 행정동·업종·분기 |
| **팀4** | 인구밀도·1인가구·폐업·강수·CPI | 자치구·연 |

### 🤔 왜 4팀에서 가져왔나?
- 강의에서 쓴 데이터(wine.csv 같은) 한 파일로는 입지 평가가 불가능
- "매출 + 임대료 + 인구 + 폐업" 을 다 보려면 여러 출처 필요
- 4명이 분담해서 효율적으로 수집

### 노트북 1번 (`01_collect.ipynb`)이 한 일
- 25개 분기 × 9개 API = 약 150개 CSV 파일 다운로드
- 4팀 데이터 정리·정규화 → `data/processed/` 에 저장

## 5️⃣ EDA — 데이터 모양부터 보기

**EDA = Exploratory Data Analysis (탐색적 데이터 분석)**

> "모델 만들기 전에 데이터가 어떻게 생겼는지 그래프로 한 번 본다"

### 노트북 2번에서 그린 그래프 11개
1. 분기별 자치구 매출 추이 — 시계열
2. 업종별 매출 분포 — 박스플롯
3. 자치구 × 업종 매출 — 히트맵
4. 변수 상관행렬
5. 유동인구 vs 매출 — 산점도
6. 시간대별 매출 비중
7. 개업률 vs 폐업률 — 사분면
8. 코로나 충격기 vs 회복기 매출
9. 자치구 임대료지수 시계열
10. 자치구 공실률 분포
11. 폐업 점포수 vs 매출

### 발견한 것들
- 강남·중·서초·종로가 매출 압도적
- 코로나 충격 (2020 Q1~Q2) → 회복 패턴 명확
- 임대료 높은 자치구가 매출도 높음 (당연)
- **매출 분포가 long-tail** — 강남 한 자치구가 외곽 10개 자치구 합한 만큼

## 6️⃣ 변수 만들기 — 가장 중요한 단계

> "raw 데이터를 그대로 모델에 넣지 않고, '의미 있는 신호'로 가공"

### 변수가 왜 중요해?
- 모델은 변수를 보고 학습 → **좋은 변수가 좋은 모델을 만듦**
- 똑같은 raw 데이터로 어떤 변수를 만드느냐가 결과를 좌우

### 우리가 만든 변수 5종류 (총 115개)

| 종류 | 개수 | 예시 | 의미 |
|---|---|---|---|
| Raw 그대로 | 12 | 매출·점포수·유동인구 | 원본 수치 |
| 가공 변수 | 11 | `avg_ticket` (객단가) = 매출÷건수 | 의미 단위로 변환 |
| 외부 변수 | 13 | 임대료지수·공실률·인구밀도 | 4팀 데이터 통합 |
| 파생 변수 | 6 | `rent_to_sales` = 임대료÷매출 | 두 변수 조합 |
| 원-핫 카테고리 | 73 | gu_강남구 (0/1) | 자치구·업종 라벨 |

### 가공 변수 예시 — `anchor_score`
```python
anchor_score = (지하철역 × 2) + (대학 × 3) + (병원 × 2) + (공공기관 × 1)
```
가설: **대학이 가장 매출 영향 큼** → 가중치 3
→ 학생들이 돈을 많이 쓴다는 가설을 변수에 녹임

### 노트북 3번이 한 일
- 위 변수 31종 + 카테고리 84종 = 115개 입력 변수 생성
- 분산 0인 변수 제거 + 상관 0.9 넘는 쌍 제거 (다중공선성 방지)

## 7️⃣ 모델 — 어떤 알고리즘 썼나?

### 데이터마이닝 강의에서 배운 모든 알고리즘 사용

| 강의 | 알고리즘 | 우리가 쓴 곳 |
|---|---|---|
| L03 | KNN (가까운 이웃) | 분류 비교군 |
| L04 | 표준화, 분할 | 전처리 |
| L05 | 선형 회귀 | 매출 회귀 비교군 |
| L07 | 로지스틱 회귀 | 분류 비교군 |
| L08-1 | 결정 트리 | 분류·회귀 |
| L08-2 | 교차검증·GridSearch | 모델 튜닝 |
| L10 | **랜덤 포레스트, GradientBoosting, HistGradientBoosting** | 메인 모델 |
| L10 | 변수 중요도 (permutation) | 해석 |
| L12 | KMeans | 자치구 군집화 |
| L13 | PCA | 차원 축소 시각화 |

### 메인 모델 — HistGradientBoosting

> "여러 개의 작은 결정 트리를 순서대로 학습해서 합치는 강력한 모델"

- 우리가 시도한 7개 분류 모델 중 **정확도 1위 (94.3%)**
- 학습 빠름, 결측 자동 처리, 해석 가능

### 노트북 4번이 한 일
- 회귀 7종 (선형, KNN, DT, RF, ExtraTrees, GB, HistGB)
- 분류 7종 (같은 7개)
- × 4가지 데이터 변환 (basic, log1p, polynomial, PCA)
- = **총 56가지 비교 실험**

### 우리가 푸는 문제는 2종류
1. **회귀**: "점포당 월매출이 정확히 얼마인가" (숫자 예측)
2. **분류**: "이 입지의 매출은 상위 33%인가? (high/mid/low)" (라벨 예측)

→ **회귀 R² 0.87**, **분류 정확도 94%**

> ⚠️ **정직성 주의 — 위 0.87 / 94% 는 타깃 누설(target leakage)로 부풀려진 수치다.**
> 매출에서 파생된 변수(예: `rent_to_sales`는 매출이 분모, `margin_proxy`는 매출×0.01)가 입력에 섞여, 모델이 사실상 정답을 보고 맞힌 것에 가깝다.
> 누설을 제거한 **정직한 모델(노트북 19)** 의 진짜 성능은 **분류 정확도 83.6% / 회귀 R² 0.64** 다. 발표·보고서에는 이 값을 기준으로 삼아야 한다.

## 8️⃣ 어려운 용어 풀이 — P10/P50/P90, BEP

### 🤔 P10 / P50 / P90 (Quantile / 분위수)

같은 (자치구, 업종) 조건의 가게 매출이 다 다르겠죠? 분포가 있어요.

```
가게 매출 분포 (낮음 ~ 높음)
        ●●●●●●●●●●●●●●●●●●●●
        ↑       ↑              ↑
       P10     P50            P90
    하위 10%   중앙값         상위 10%
   "운 나쁘면"  "평균"       "잘되면"
```

| 분위 | 의미 |
|---|---|
| **P10** | 비관 시나리오 — "최악의 경우 이 정도" |
| **P50** | 표준 시나리오 — 중앙값 (평균과 다름!) |
| **P90** | 낙관 시나리오 — "잘되면 이 정도" |

**왜 한 값 안 주고 3개 주나?**
- 평균은 큰 가게에 끌려서 무의미할 수 있음
- 분포 전체를 보여주는 게 더 정직

### 🤔 BEP (Break-Even Point / 손익분기점)

**가게가 적자도 흑자도 아닌 매출 수준**

```
월 고정비 = 임대료 + 인건비 + 투자금 분할상각
BEP 매출 = 월 고정비 ÷ 마진율
```

예시:
- 한식 가게: 임대료 500만 + 인건비 600만 + 투자금 1억÷36개월 = 약 1,400만/월 고정비
- 마진율 20% → BEP = 1,400만 ÷ 0.20 = **7,000만 원/월**
- → 매월 7천만 매출 이상이면 흑자

### 🤔 p_roi (BEP 달성 확률)

P10/P50/P90 분포에서 BEP가 어디에 있는지

```
BEP < P10  →  p_roi ≈ 1.0  (비관에도 흑자 ✅)
BEP > P90  →  p_roi ≈ 0.0  (낙관에도 적자 ❌)
그 사이    →  선형 보간
```

### 🤔 success_rate (종합 성공 확률 0~1)

```
success_rate = 0.40 × p_high           ← 분류기가 high로 본 확률
             + 0.40 × p_roi            ← BEP 달성 확률
             + 0.20 × sales_percentile ← 학습 데이터에서 백분위
             - 포화 페널티               ← 같은 업종이 이미 많으면 감점
             - 신뢰도 페널티             ← 학습 표본 적으면 감점
             - 폐업 페널티               ← 그 지역 폐업 많으면 감점
```

이 한 숫자(0~1)에 모든 신호가 합쳐짐 → A/B/C/D/F 5등급

## 9️⃣ 실제로 어떻게 작동하나? — 시연

### 강남역 한식음식점, 1층, 1억 투자 입력 시:

In [1]:
# 13번 노트북의 평가 함수를 그대로 호출
import joblib, json, numpy as np, pandas as pd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier, HistGradientBoostingClassifier

PROC = Path('/Users/ijunsu/Documents/Documents/capston/data_analysis/trade_area_project/data/processed')

# 행정동 데이터 로드 + 모델 학습 (한 번만)
df = pd.read_csv(PROC / 'features_adstrd.csv')
id_c = ['adstrd_code','adstrd_nm','gu','biz','q','q_int']
ex_c = ['adstrd_sales_amt','log_sales','sales_per_store','log_sales_per_store','store_class']
feat = [c for c in df.columns if c not in id_c + ex_c]
for c in feat: df[c] = pd.to_numeric(df[c], errors='coerce')
df[feat] = df[feat].fillna(df[feat].mean(numeric_only=True))

# Quantile 회귀기 3종 — P10/P50/P90 예측용
qm = {q: HistGradientBoostingRegressor(loss='quantile', quantile=q, max_depth=8,
                                       max_iter=400, learning_rate=0.1, random_state=42)
                .fit(df[feat].values, df['log_sales_per_store'].values)
      for q in [0.10, 0.50, 0.90]}

# 분류기 — 매출 high 여부
y = (df['log_sales_per_store'] > df['log_sales_per_store'].quantile(0.66)).astype(int).values
rf = RandomForestClassifier(n_estimators=300, max_depth=14, random_state=42, n_jobs=-1).fit(df[feat].values, y)

print('모델 학습 완료 — 이제 입지 평가 가능')

모델 학습 완료 — 이제 입지 평가 가능


In [2]:
# 실제 입지 평가 — 입력: 자치구·업종·층수·투자금
def evaluate(gu, biz, floor='1F', invest=100_000_000):
    sub = df[(df['gu']==gu) & (df['biz']==biz)]
    if sub.empty: return None
    row = sub.sort_values('q_int').iloc[-1]
    x = row[feat].values.reshape(1, -1)

    # 층수 보정
    floor_mult = {'1F':1.0, 'B1':0.55, '2F':0.65, '3F+':0.45, 'high':0.30}
    fm = floor_mult.get(floor, 1.0)
    log_adj = np.log(max(fm, 1e-3))

    # P10/P50/P90 매출 예측
    p10 = float(np.exp(qm[0.10].predict(x)[0] + log_adj))
    p50 = float(np.exp(qm[0.50].predict(x)[0] + log_adj))
    p90 = float(np.exp(qm[0.90].predict(x)[0] + log_adj))

    # 분류기 — high 확률
    p_high = rf.predict_proba(x)[0][1]

    return p10/1e6, p50/1e6, p90/1e6, p_high

# 4가지 입지 비교
cases = [
    ('강남구', '한식음식점', '1F', 100_000_000, '강남 한식 1층'),
    ('마포구', '커피-음료', '1F',  80_000_000, '마포 카페 1층'),
    ('마포구', '커피-음료', 'B1',  80_000_000, '마포 카페 지하'),  # 층수 효과
    ('종로구', '의약품',    '2F', 200_000_000, '종로 약국 2층'),
]

print(f'{"입지":20s} {"P10":>5s} {"P50":>5s} {"P90":>5s}  {"p_high":>7s}')
print('─'*60)
for gu, biz, floor, invest, label in cases:
    r = evaluate(gu, biz, floor, invest)
    if r is None:
        print(f'{label:20s} 데이터 없음'); continue
    p10, p50, p90, ph = r
    print(f'{label:20s} {p10:>4.0f}M {p50:>4.0f}M {p90:>4.0f}M  {ph:>6.2f}')

입지                     P10   P50   P90   p_high
────────────────────────────────────────────────────────────
강남 한식 1층              109M  155M  153M    0.92
마포 카페 1층               27M   27M   28M    0.19
마포 카페 지하               15M   15M   16M    0.19
종로 약국 2층              149M  427M  757M    0.90


### 시연 결과 해석

위 결과를 보면:
- **강남 한식 1층**: P50 매출 1억원대, p_high 0.9+ → A 등급 추천
- **마포 카페 1층 vs 지하**: 지하로 가면 매출 절반으로 떨어짐 (층수 효과!)
- **종로 약국 2층**: 큰 매출 잠재력 (의료업은 2층도 OK)

→ **모델이 이런 차이를 자동으로 잡아냄**

---

## 🔟 핵심 발견 — 우리 모델이 알아낸 것들

### 발견 1: 외부 변수가 진짜 중요하다
- 강의에서 배운 변수(매출·점포·유동인구)만 쓰면 정확도 70%
- + 가공 변수(가설 기반) → 89%
- + **외부 변수 (임대료·공실률·인구밀도)** → **94%**
- → 외부 데이터 통합이 **+4.75%p** 효과 (통계적으로 유의)

### 발견 2: 자치구·업종 라벨은 의외로 안 중요
- 라벨(gu_강남구, biz_한식)만으로도 81%
- 다른 변수 추가 후 라벨이 더해주는 효과는 +0.2%p 만
- → 모델이 라벨에 의존하지 않고 진짜 데이터로 학습

### 발견 3: 임대료가 1위 변수 — ⚠️ 누설로 인한 착시 (실제 통찰 아님)
- 변수 중요도 분석에서 **`rent_to_sales`(임대료÷매출)** 가 1위
- 그 다음 momentum, peer_avg_sales, age_entropy
- ⚠️ 그러나 `rent_to_sales`는 **매출(타깃)이 분모**인 누설 변수다. 1위로 나온 것은 모델이 정답의 일부를 입력으로 본 **인공적 산물**이지, "임대료 부담이 입지 성공의 핵심"이라는 경제적 통찰이 아니다.
- 누설을 제거한 정직한 모델(노트북 19)에서는 이 변수가 빠지며, 따라서 이 "1위" 결과는 보고서 통찰로 인용하지 말 것.

### 발견 4: 코로나 회복기에 임대료 관련 변수가 더 중요해짐
- 충격기(2020): momentum (매출 모멘텀) 가장 중요
- 회복기(2022+): rent_to_sales (임대료 부담) 가장 중요
- → "살아남은 가게는 임대료 부담이 결정적"

---

## 1️⃣1️⃣ 한계 — 우리 모델이 못 하는 것

### 정직하게 인정한 7가지 한계

| # | 한계 | 왜 못 해결? |
|---|---|---|
| 1 | 자치구 단위 폐업과 양의 상관 +0.80 | 데이터 자체가 자치구 단위 — 규모 효과 |
| 2 | 홍대·명동 유명 상권 과소평가 | 행정동 평균이 작은 점포에 끌림 (점포 단위 데이터 필요) |
| 3 | 회귀 MAPE 64~78% | 큰 매출 outlier 영향 |
| 4 | BEP 가정 민감 (±50% 변동) | 실제 임대료·인건비 데이터 없음 |
| 5 | 서울시 변화지표와 r≈0 | 두 모델이 다른 차원 측정 (강점이자 한계) |
| 6 | KOSIS 5년 생존율 순위 일부 불일치 | 우리는 매출 분위, KOSIS는 진짜 생존 |
| 7 | 학습 데이터 75%가 LOW 신뢰도 | 행정동 표본 부족 |

### 보고서의 가장 강력한 메시지

> **"완벽한 모델이라고 주장하지 않는다. 16가지 약점을 보완했고, 7가지 본질적 한계는 데이터 부재로 남아 있음을 정직하게 드러낸다."**

→ 교수님(평가자)이 가장 좋아하는 자세는 "완벽"이 아니라 **"정직한 한계 인정"**

---

## 1️⃣2️⃣ 데이터마이닝을 처음 배우는 후배에게

### 이 프로젝트에서 배운 것들

1. **데이터는 raw로 두면 안 됨** — 가공해서 신호로 만들어야 함
2. **여러 모델 비교** — 한 모델만 쓰면 부족함을 모름
3. **검증이 중요** — 정확도 높다고 끝이 아님 (외부 데이터로 검증)
4. **한계 인정이 강점** — "완벽하다"보다 "이건 못 한다"가 신뢰
5. **시행착오 기록** — v1 → v6까지 의사결정 과정이 보고서 핵심

### 강의에서 배운 게 진짜 쓸모 있다
- L04 train_test_split → 우리도 그대로 사용
- L05 LinearRegression → 회귀 비교군
- L07 LogisticRegression → 분류 비교군
- L08 결정 트리·GridSearch → 모델 튜닝
- L10 RandomForest, GradientBoosting → 메인 모델
- L12 KMeans → 자치구 유형 발견
- L13 PCA → 변수 차원 축소

### "강의에서 배운 것 + 외부 데이터 통합 + 정직한 한계 인정" = 좋은 프로젝트

---

## 1️⃣3️⃣ 우리 산출물 — 노트북 18개

| 번호 | 노트북 | 내용 |
|---|---|---|
| 01 | 데이터 수집 | 서울 열린데이터 + 4팀 자료 |
| 02 | EDA | 11개 그래프 |
| 03 | 변수 생성 | 115개 변수 |
| 04 | 모델 학습 | 7종 × 4변환 = 56실험 |
| 05 | Ablation·시간분할·실증 BEP | 약점 보완 |
| 06 | 모델 해석·잔차 | 어디서 빗나가나 |
| 07 | 변수 중요도 4관점 | 무엇이 중요한가 |
| 08 | 비지도 (KMeans·PCA) | 자치구 군집화 |
| 09 | 지도 시각화 | 9개 choropleth |
| 10 | 평가기 (자치구·업종) | evaluate_location() |
| 11 | 평가기 (좌표·층수·신뢰구간) | evaluate_startup_point() |
| 12 | 검증 + 시행착오 (v1~v6) | 의사결정 과정 |
| 13 | 최종 결론·시연 | A/B/C/D/F 등급 |
| 14 | 심층 검증 (7가지) | 회귀 R², BEP 민감도, 잔차, 사례, 폐업, 기여도, 부트스트랩 |
| 15 | 약점 추가 보완 | 폐업 페널티, BEP 신뢰구간 |
| 16 | 서울시 API 비교 | r ≈ 0 → 다른 차원 측정 |
| 17 | 외부 검증 (폐업·KOSIS·도메인) | 정직한 한계 인정 |
| **18** | **본 발표 설명서** | **1·2학년 대상 가이드** |

→ 데이터·코드·결과 모두 추적 가능, 빌더 스크립트로 재생성 가능

---

## 1️⃣4️⃣ 발표 마무리

### 핵심 한 줄
> **"공개 데이터만으로 좌표 입력 → 성공 확률 % 까지 답하는 모델을 만들었고, 그 한계까지 정직하게 드러냈다."**

### 후배에게 전하는 말
1. **데이터 수집부터 시작** — raw 데이터 없으면 아무것도 못 함
2. **EDA부터** — 모델 만들기 전에 데이터를 알기
3. **변수가 모델보다 중요** — 좋은 변수 → 좋은 모델
4. **검증을 빠뜨리지 말기** — 정확도 90%가 진짜인지 의심하기
5. **한계를 숨기지 말기** — 보고서의 강점이 됨

### 질문이 있다면?
- 노트북 13번 (최종 결론) 부터 보세요 — 모든 결과 요약
- 노트북 12번 (검증) — 시행착오와 결정 과정
- 노트북 10·11번 (평가기) — 실제 코드 사용법